# sc1_gan_preupsample Evaluation

This notebook evaluates trained Scenario-1 (Forecast Correction) GAN Pre-Upsample runs and computes metrics using:

- `MAE = (1/(N*L*W)) * sum(|S - O|)`
- `RMSE = sqrt((1/(N*L*W)) * sum((S - O)^2))`

where `N` is number of test samples, `L` is latitude size, and `W` is longitude size.


In [1]:
# ==============================
# Inputs
# ==============================
from pathlib import Path

# All 5 runs for this model
RUN_DIRS = [
    "runs TA/sc1_gan_preupsample/20260413_154641_lead1d",
    "runs TA/sc1_gan_preupsample/20260413_171329_lead1d",
    "runs TA/sc1_gan_preupsample/20260413_184014_lead1d",
    "runs TA/sc1_gan_preupsample/20260413_200656_lead1d",
    "runs TA/sc1_gan_preupsample/20260413_213340_lead1d",
]

# Scenario-1 data files
FORECAST_PATH = "data/ifs_lowres_indonesia_2018-2022.zarr"
TRUTH_PATH    = "data/era5_indonesia_2018-2022.zarr"

# Optional override (None = read from run config.json)
TEST_START_DATE_OVERRIDE = None
TEST_END_DATE_OVERRIDE   = None

# Variables in fixed order
VARS = [
    "10m_u_component_of_wind",
    "10m_v_component_of_wind",
    "2m_temperature",
    "total_precipitation_24hr",
]
VAR_LABELS = ["U10 (m/s)", "V10 (m/s)", "T2m (K)", "TP 24hr (mm)"]
TP_IDX = VARS.index("total_precipitation_24hr")

# Validate first run directory exists
first_run = Path(RUN_DIRS[0])
if not first_run.exists():
    raise FileNotFoundError(
        f"First run dir not found: {first_run}. "
        f"Ensure runs TA/ directory is in the working directory."
    )

print(f"Configured {len(RUN_DIRS)} runs for evaluation")
for rd in RUN_DIRS:
    print(f"  {rd}")


Configured 5 runs for evaluation
  runs TA/sc1_gan_preupsample/20260413_154641_lead1d
  runs TA/sc1_gan_preupsample/20260413_171329_lead1d
  runs TA/sc1_gan_preupsample/20260413_184014_lead1d
  runs TA/sc1_gan_preupsample/20260413_200656_lead1d
  runs TA/sc1_gan_preupsample/20260413_213340_lead1d


In [9]:
# ==============================
# Imports + run config (from first run)
# ==============================
import json
import os
import csv
import numpy as np
import pandas as pd
import xarray as xr
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# Read config from first run (all runs share same hyperparameters)
CFG_PATH = Path(RUN_DIRS[0]) / "config.json"

cfg = {}
if CFG_PATH.exists():
    cfg = json.loads(CFG_PATH.read_text(encoding="utf-8"))

lead_days = int(cfg.get("lead_days", 1))
scale = int(cfg.get("scale", 6))

test_start_date = TEST_START_DATE_OVERRIDE or cfg.get("test_start_date", "2022-07-01")
test_end_date   = TEST_END_DATE_OVERRIDE   or cfg.get("test_end_date", "2022-12-31")

print("lead_days:", lead_days)
print("scale:", scale)
print("test range:", test_start_date, "->", test_end_date)

from torch.utils.data import TensorDataset, DataLoader


lead_days: 1
scale: 6
test range: 2022-07-01 -> 2022-12-31


In [3]:
# ==========================================
# ==========================================
# GAN Architecture for Super-Resolution  
# Based on ESRGAN with RRDB blocks
# ==========================================
# ==========================================
# ==========================================
# GAN Architecture for Super-Resolution  
# Based on ESRGAN with RRDB blocks
# ==========================================

# ==========================================
# Dense Block (used in RRDB)
# ==========================================

class DenseLayer(nn.Module):
    """Single dense layer: Conv + ReLU"""
    def __init__(self, in_ch, growth_rate=32):
        super().__init__()
        self.conv = nn.Conv2d(in_ch, growth_rate, kernel_size=3, padding=1, bias=True)
        self.relu = nn.ReLU(inplace=True)
    
    def forward(self, x):
        out = self.relu(self.conv(x))
        return torch.cat([x, out], dim=1)  # Concatenate input with output


class ResidualDenseBlock(nn.Module):
    """
    Residual Dense Block (RDB)
    Multiple dense layers with residual connection
    """
    def __init__(self, in_ch, growth_rate=32, num_layers=5):
        super().__init__()
        self.layers = nn.ModuleList()
        for i in range(num_layers):
            self.layers.append(DenseLayer(in_ch + i * growth_rate, growth_rate))
        
        # Local fusion (1x1 conv to reduce channels back)
        self.local_fusion = nn.Conv2d(
            in_ch + num_layers * growth_rate,
            in_ch,
            kernel_size=1,
            bias=True
        )
        self.beta = 0.2  # Residual scaling factor
    
    def forward(self, x):
        identity = x
        for layer in self.layers:
            x = layer(x)
        x = self.local_fusion(x)
        return identity + self.beta * x


class RRDB(nn.Module):
    """
    Residual-in-Residual Dense Block
    Stack of multiple RDB blocks with residual connection
    """
    def __init__(self, in_ch, growth_rate=32, num_rdb=3):
        super().__init__()
        self.rdb_blocks = nn.ModuleList([
            ResidualDenseBlock(in_ch, growth_rate) for _ in range(num_rdb)
        ])
        self.beta = 0.2  # Residual scaling factor
    
    def forward(self, x):
        identity = x
        for rdb in self.rdb_blocks:
            x = rdb(x)
        return identity + self.beta * x


# ==========================================
# Generator Network (RRDB-based)
# ==========================================

class RRDBGenerator(nn.Module):
    """
    Generator with RRDB blocks for super-resolution
    Input  : (B, 4, 24, 32)   — low-res forecast  
    Output : (B, 4, 144, 192) — high-res prediction
    """
    def __init__(self, in_ch=4, out_ch=4, base_ch=64, num_rrdb=4, growth_rate=32):
        super().__init__()
        
        # Initial feature extraction
        self.conv_first = nn.Conv2d(in_ch, base_ch, kernel_size=3, padding=1, bias=True)
        
        # RRDB blocks
        self.rrdb_blocks = nn.ModuleList([
            RRDB(base_ch, growth_rate=growth_rate) for _ in range(num_rrdb)
        ])
        
        # Feature fusion after RRDB blocks
        self.conv_body = nn.Conv2d(base_ch, base_ch, kernel_size=3, padding=1, bias=True)
        
        # 6× Upsampling (via two 2× steps + one 3× step)
        # Step 1: 2× upscale  (24,32) → (48,64)
        self.up1 = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            nn.Conv2d(base_ch, base_ch, kernel_size=3, padding=1, bias=True),
            nn.ReLU(inplace=True)
        )
        
        # Step 2: 3× upscale (48,64) → (144,192)
        self.up2 = nn.Sequential(
            nn.Upsample(scale_factor=3, mode='bilinear', align_corners=False),
            nn.Conv2d(base_ch, base_ch, kernel_size=3, padding=1, bias=True),
            nn.ReLU(inplace=True)
        )
        
        # Final refinement
        self.conv_last = nn.Sequential(
            nn.Conv2d(base_ch, base_ch, kernel_size=3, padding=1, bias=True),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_ch, out_ch, kernel_size=3, padding=1, bias=True)
        )
    
    def forward(self, x):
        # Initial feature extraction
        feat_init = self.conv_first(x)  # (B, 64, 24, 32)
        
        # RRDB trunk
        feat = feat_init
        for rrdb in self.rrdb_blocks:
            feat = rrdb(feat)
        
        # Global residual learning
        feat = self.conv_body(feat)
        feat = feat + feat_init  # Long skip connection
        
        # 6× upsampling
        feat = self.up1(feat)  # (B, 64, 48, 64)
        feat = self.up2(feat)  # (B, 64, 144, 192)
        
        # Final output
        out = self.conv_last(feat)  # (B, 4, 144, 192)
        return out
        
class ResidualBlock(nn.Module):
    """
    Mirrors dl4ds ResidualBlock: Conv → Norm → Act → Conv → Norm + skip
    """
    def __init__(self, n_filters, normalization=None, activation='relu'):
        super().__init__()
        layers_1 = [nn.Conv2d(n_filters, n_filters, 3, padding=1, bias=(normalization is None))]
        if normalization == 'bn':
            layers_1.append(nn.BatchNorm2d(n_filters))
        elif normalization == 'ln':
            layers_1.append(nn.GroupNorm(1, n_filters))  # LayerNorm equivalent for 2D
        layers_1.append(nn.ReLU(inplace=True) if activation == 'relu' else nn.LeakyReLU(0.2, inplace=True))

        layers_2 = [nn.Conv2d(n_filters, n_filters, 3, padding=1, bias=(normalization is None))]
        if normalization == 'bn':
            layers_2.append(nn.BatchNorm2d(n_filters))
        elif normalization == 'ln':
            layers_2.append(nn.GroupNorm(1, n_filters))

        self.block1 = nn.Sequential(*layers_1)
        self.block2 = nn.Sequential(*layers_2)
        self.act_out = nn.ReLU(inplace=True) if activation == 'relu' else nn.LeakyReLU(0.2, inplace=True)

    def forward(self, x):
        identity = x
        out = self.block1(x)
        out = self.block2(out)
        return self.act_out(out + identity)


# ==========================================
# Conditional Discriminator — dl4ds style
# ==========================================

class ResidualDiscriminator(nn.Module):
    """
    Matches dl4ds residual_discriminator() structure:

    Branch 1 (LR input):
        Conv → n_res_blocks × ResidualBlock → Conv → Add(skip)

    Branch 2 (HR reference/generated):
        Conv → n_res_blocks × ResidualBlock → downsample to LR size

    Fusion:
        Concat → ResidualBlock → GlobalAvgPool → Dropout → Dense(32) → Dense(1)

    Args:
        in_ch_lr      : channels in LR input  (matches your in_ch_lr=4)
        in_ch_hr      : channels in HR input  (matches your in_ch_hr=4)
        lr_size       : (H, W) of LR grid, e.g. (24, 32) — used to downsample HR branch
        scale         : upsampling scale factor (e.g. 6)
        n_filters     : base filter count (dl4ds default: 8, feel free to use 64)
        n_res_blocks  : residual blocks per branch (dl4ds default: 4)
        normalization : None | 'bn' | 'ln'
        activation    : 'relu' | 'leaky'
    """
    def __init__(
        self,
        in_ch_lr=4,
        in_ch_hr=4,
        lr_size=(32, 24),
        scale=6,
        n_filters=64,
        n_res_blocks=4,
        normalization=None,
        activation='relu',
    ):
        super().__init__()
        self.lr_size = lr_size
        self.scale = scale
        act = activation

        # --------------------------------------------------
        # Branch 1: LR input
        # --------------------------------------------------
        self.lr_stem = nn.Conv2d(in_ch_lr, n_filters, 3, padding=1, bias=True)
        self.lr_res_blocks = nn.ModuleList([
            ResidualBlock(n_filters, normalization=normalization, activation=act)
            for _ in range(n_res_blocks)
        ])
        self.lr_conv_end = nn.Conv2d(n_filters, n_filters, 3, padding=1, bias=True)

        # --------------------------------------------------
        # Branch 2: HR reference / generated
        # --------------------------------------------------
        self.hr_stem = nn.Conv2d(in_ch_hr, n_filters, 3, padding=1, bias=True)
        self.hr_res_blocks = nn.ModuleList([
            ResidualBlock(n_filters, normalization=normalization, activation=act)
            for _ in range(n_res_blocks)
        ])

        # Downsample HR → LR spatial size (mirrors dl4ds post-upsampling path)
        # scale=4 → stride-2 conv ×2; scale=6 → bilinear resize (like dl4ds 'else' branch)
        if scale == 4:
            self.hr_downsample = nn.Sequential(
                nn.Conv2d(n_filters, n_filters, 3, padding=1, stride=2, bias=False),
                nn.Conv2d(n_filters, n_filters, 3, padding=1, stride=2, bias=False),
            )
        else:
            # General case: let interpolation handle arbitrary scales
            self.hr_downsample = None   # handled in forward() with F.interpolate

        # --------------------------------------------------
        # Fusion head  (matches dl4ds concat → ResBlock → GAP → Drop → Dense)
        # --------------------------------------------------
        fused_ch = n_filters * 2          # after Concatenate([branch1, branch2])
        self.fusion_res = ResidualBlock(
            # dl4ds uses x.shape[-1] which is fused_ch — we need a projection first
            # because ResidualBlock expects in==out; use a 1×1 to align
            n_filters,
            normalization=normalization,
            activation=act,
        )
        self.fusion_proj = nn.Conv2d(fused_ch, n_filters, 1, bias=True)  # 1×1 projection

        self.gap = nn.AdaptiveAvgPool2d(1)   # GlobalAveragePooling2D
        self.dropout = nn.Dropout(0.4)
        self.fc1 = nn.Linear(n_filters, 32)
        self.fc2 = nn.Linear(32, 1)

    # ----------------------------------------------------------
    def forward(self, x_lr, x_hr):
        # ---------- Branch 1: LR ----------
        x_1 = self.lr_stem(x_lr)                    # (B, F, H_lr, W_lr)
        b = x_1
        for blk in self.lr_res_blocks:
            b = blk(b)
        b = self.lr_conv_end(b)
        x_1 = x_1 + b                               # residual add (mirrors dl4ds Add)

        # ---------- Branch 2: HR → downsample ----------
        x_2 = self.hr_stem(x_hr)                    # (B, F, H_hr, W_hr)
        for blk in self.hr_res_blocks:
            x_2 = blk(x_2)

        if self.hr_downsample is not None:
            x_2 = self.hr_downsample(x_2)
        else:
            # Bilinear resize to LR spatial dims (mirrors dl4ds InterpolationDownsampling)
            x_2 = F.interpolate(
                x_2,
                size=self.lr_size,
                mode='bilinear',
                align_corners=False,
            )

        # ---------- Fusion ----------
        x = torch.cat([x_1, x_2], dim=1)            # (B, 2F, H_lr, W_lr)
        x = self.fusion_proj(x)                      # (B, F,  H_lr, W_lr)  — 1×1 projection
        x = self.fusion_res(x)                       # ResidualBlock on fused features

        x = self.gap(x)                              # (B, F, 1, 1)
        x = self.dropout(x)
        x = x.flatten(1)                             # (B, F)
        x = torch.sigmoid(self.fc1(x))              # Dense(32, activation='sigmoid')
        x = torch.sigmoid(self.fc2(x))              # Dense(1,  activation='sigmoid')
        return x


# Quick sanity check
print("Testing GAN Architecture...")
gen = RRDBGenerator(in_ch=4, out_ch=4, base_ch=64, num_rrdb=4)
disc = ResidualDiscriminator(in_ch_lr=4, in_ch_hr=4, lr_size=(24, 32))

dummy_lr = torch.randn(2, 4, 24, 32)
dummy_hr = torch.randn(2, 4, 144, 192)

output_hr = gen(dummy_lr)
output_score = disc(dummy_lr, dummy_hr)

print(f"\n✓ Generator:")
print(f"  Input  : {dummy_lr.shape}")
print(f"  Output : {output_hr.shape}")
print(f"  Params : {sum(p.numel() for p in gen.parameters() if p.requires_grad):,}")

print(f"\n✓ Discriminator:")
print(f"  Input  : {dummy_hr.shape}")
print(f"  Output : {output_score.shape}")
print(f"  Params : {sum(p.numel() for p in disc.parameters() if p.requires_grad):,}")

Testing GAN Architecture...

✓ Generator:
  Input  : torch.Size([2, 4, 24, 32])
  Output : torch.Size([2, 4, 144, 192])
  Params : 2,538,948

✓ Discriminator:
  Input  : torch.Size([2, 4, 144, 192])
  Output : torch.Size([2, 1])
  Params : 716,737


In [4]:
# ==============================
# Load Scenario-1 data + align + split test
# ==============================
ds_forecast = xr.open_dataset(FORECAST_PATH)
ds_truth = xr.open_dataset(TRUTH_PATH)

lead_td = np.timedelta64(lead_days, "D")
ds_forecast = ds_forecast.sel(prediction_timedelta=lead_td)

# Sort latitude for stable slicing
ds_truth = ds_truth.sortby("latitude")
ds_forecast = ds_forecast.sortby("latitude")

# Spatial crop logic (same as training notebook)
tr_lons = ds_truth.longitude.values
tr_lats = ds_truth.latitude.values
fc_lons = ds_forecast.longitude.values
fc_lats = ds_forecast.latitude.values

valid_lons = fc_lons[(fc_lons >= tr_lons.min()) & (fc_lons <= tr_lons.max())]
valid_lats = fc_lats[(fc_lats >= tr_lats.min()) & (fc_lats <= tr_lats.max())]

lon_start = valid_lons[0]
lat_start = valid_lats[0]

lon_start_idx = np.argmin(np.abs(tr_lons - lon_start))
lat_start_idx = np.argmin(np.abs(tr_lats - lat_start))

avail_lon = len(tr_lons) - lon_start_idx
avail_lat = len(tr_lats) - lat_start_idx
max_fc_lon = min(avail_lon // scale, 32)
max_fc_lat = min(avail_lat // scale, 24)

fc_lon_start_idx = np.argmin(np.abs(fc_lons - lon_start))
fc_lat_start_idx = np.argmin(np.abs(fc_lats - lat_start))

ds_fc = ds_forecast.isel(
    longitude=slice(fc_lon_start_idx, fc_lon_start_idx + max_fc_lon),
    latitude=slice(fc_lat_start_idx, fc_lat_start_idx + max_fc_lat),
)

ds_tr = ds_truth.isel(
    longitude=slice(lon_start_idx, lon_start_idx + len(ds_fc.longitude) * scale),
    latitude=slice(lat_start_idx, lat_start_idx + len(ds_fc.latitude) * scale),
)

# Temporal alignment
valid_time = ds_fc.time + lead_td
common_times = np.intersect1d(valid_time.values, ds_tr.time.values)

ds_fc = ds_fc.assign_coords(valid_time=valid_time).sel(valid_time=common_times)
ds_fc = ds_fc.assign_coords(time=ds_fc.valid_time).drop_vars("valid_time")
ds_tr_aligned = ds_tr.sel(time=common_times)

# Build arrays
X = np.stack([ds_fc[v].values for v in VARS], axis=1).astype(np.float32)   # (N,C,24,32)
Y = np.stack([ds_tr_aligned[v].values for v in VARS], axis=1).astype(np.float32)  # (N,C,144,192)

# Fill NaNs in X per-channel
for c in range(X.shape[1]):
    ch = X[:, c, :, :]
    mask = np.isnan(ch)
    if mask.any():
        fill_val = np.nanmean(ch)
        ch[mask] = fill_val
        X[:, c, :, :] = ch

# Axis fix (if needed)
expected_h = X.shape[2] * scale
expected_w = X.shape[3] * scale
if Y.shape[2] == expected_w and Y.shape[3] == expected_h:
    Y = np.transpose(Y, (0, 1, 3, 2))

# Log1p precipitation channel (same preprocessing as training)
X[:, TP_IDX] = np.log1p(np.clip(X[:, TP_IDX], 0, None))
Y[:, TP_IDX] = np.log1p(np.clip(Y[:, TP_IDX], 0, None))

# Date-based test split
times = ds_fc.time.values
test_mask = (times >= np.datetime64(test_start_date)) & (times <= np.datetime64(test_end_date))
assert test_mask.any(), "No test samples found for selected test date range"

X_test = X[test_mask]
Y_test = Y[test_mask]
print("X_test shape:", X_test.shape)
print("Y_test shape:", Y_test.shape)


# Save raw test arrays for multi-run evaluation
X_test_raw = X_test.copy()
Y_test_raw = Y_test.copy()
print("Raw test arrays saved for multi-run evaluation.")


/tmp/ipykernel_14168/2134806482.py:4: FutureWarning: In a future version, xarray will not decode the variable 'prediction_timedelta' into a timedelta64 dtype based on the presence of a timedelta-like 'units' attribute by default. Instead it will rely on the presence of a timedelta64 'dtype' attribute, which is now xarray's default way of encoding timedelta64 values.
To continue decoding into a timedelta64 dtype, either set `decode_timedelta=True` when opening this dataset, or add the attribute `dtype='timedelta64[ns]'` to this variable on disk.
To opt-in to future behavior, set `decode_timedelta=False`.
  ds_forecast = xr.open_dataset(FORECAST_PATH)


X_test shape: (367, 4, 32, 24)
Y_test shape: (367, 4, 192, 144)
Raw test arrays saved for multi-run evaluation.


In [ ]:
# ==============================
# Helper functions + baseline (computed once)
# ==============================

def denormalize(tensor, mu, sig, tp_idx=TP_IDX):
    """Reverse z-score, then expm1 on the TP channel."""
    mu  = mu.to(tensor.device)
    sig = sig.to(tensor.device)
    out = tensor * (sig + 1e-6) + mu
    out[:, tp_idx] = torch.expm1(out[:, tp_idx])
    return out


def compute_metrics(preds_norm, targets_norm, Y_mu, Y_sig):
    """
    Denormalize both tensors, then compute per-variable
    RMSE, MAE, Bias, and Corr over the full (N, C, H, W) batch.
    """
    p_dn = denormalize(preds_norm.clone(), Y_mu, Y_sig).numpy()
    t_dn = denormalize(targets_norm.clone(), Y_mu, Y_sig).numpy()

    if p_dn.shape[-2:] == t_dn.shape[-2:][::-1]:
        t_dn = np.transpose(t_dn, (0, 1, 3, 2))
    elif p_dn.shape[-2:] != t_dn.shape[-2:]:
        raise ValueError(f"Prediction/target spatial mismatch: pred={p_dn.shape}, target={t_dn.shape}")

    results = {}
    for vi, label in enumerate(VAR_LABELS):
        p = p_dn[:, vi]; t = t_dn[:, vi]
        diff = p - t
        rmse = float(np.sqrt(np.nanmean(diff ** 2)))
        mae  = float(np.nanmean(np.abs(diff)))
        bias = float(np.nanmean(diff))
        pd_ = p - np.nanmean(p)
        td_ = t - np.nanmean(t)
        with np.errstate(divide='ignore', invalid='ignore'):
            corr = float(
                np.nansum(pd_ * td_) /
                (np.sqrt(np.nansum(pd_**2) * np.nansum(td_**2)) + 1e-12)
            )
        results[label] = {'RMSE': rmse, 'MAE': mae, 'Bias': bias, 'Corr': corr}
    return results


def mae_formula_13(pred, obs):
    pred = np.asarray(pred, dtype=np.float64)
    obs  = np.asarray(obs, dtype=np.float64)
    N, L, W = pred.shape
    return float(np.sum(np.abs(pred - obs)) / (N * L * W))


def rmse_formula_14(pred, obs):
    pred = np.asarray(pred, dtype=np.float64)
    obs  = np.asarray(obs, dtype=np.float64)
    N, L, W = pred.shape
    return float(np.sqrt(np.sum((pred - obs) ** 2) / (N * L * W)))


# ── Compute baseline (common across all runs) ──────────────────────────
fc_test_lowres = ds_fc.sel(time=slice(test_start_date, test_end_date)).copy(deep=True)
tr_test_ds     = ds_tr_aligned.sel(time=slice(test_start_date, test_end_date))

for v_name in VARS:
    arr = fc_test_lowres[v_name].values
    if np.isnan(arr).all():
        raise ValueError(f"All NaN in baseline low-res '{v_name}' for test period")
    if np.isnan(arr).any():
        fc_test_lowres[v_name] = fc_test_lowres[v_name].fillna(float(np.nanmean(arr)))

baseline_rmse_by_var = {}
for v_name in VARS:
    low_da    = fc_test_lowres[v_name].transpose('time', 'latitude', 'longitude')
    truth_da  = tr_test_ds[v_name].transpose('time', 'latitude', 'longitude')
    common_t  = np.intersect1d(low_da.time.values, truth_da.time.values)
    if common_t.size == 0:
        raise ValueError(f"No overlapping time steps for baseline: {v_name}")
    low_da   = low_da.sel(time=common_t)
    truth_da = truth_da.sel(time=common_t)
    base_da  = low_da.interp(latitude=truth_da.latitude,
                             longitude=truth_da.longitude, method='linear')
    rmse_grid = np.sqrt(np.mean((base_da.values - truth_da.values) ** 2, axis=0))
    baseline_rmse_by_var[v_name] = float(np.nanmean(rmse_grid))

print("Baseline RMSE (bilinear interpolation):")
for v, r in baseline_rmse_by_var.items():
    print(f"  {v}: {r:.4f}")
print("\nHelper functions defined. Ready for evaluation loop.")

In [ ]:
# ==============================
# Evaluation loop — all SC1 GAN Pre-Up runs
# ==============================
target_date    = np.datetime64('2022-08-01')
VAR_LABELS_VIZ = VAR_LABELS.copy()
BATCH_SIZE     = 128
BASELINE_EPS   = 1e-8

# Coordinate arrays for plotting
lons_lr = ds_fc.longitude.values
lats_lr = ds_fc.latitude.values
lons_hr = ds_tr_aligned.longitude.values
lats_hr = ds_tr_aligned.latitude.values

# Hardcoded color ranges
FIELD_VMIN = [      -6,    -8,   285,   0.000]
FIELD_VMAX = [       6,     8,   302,   0.025]
ERROR_LIM  = [       4,     4,     5,   0.015]
cmaps_field = ["RdBu_r", "RdBu_r", "plasma", "YlGnBu"]

test_times = ds_tr_aligned.sel(time=slice(test_start_date, test_end_date)).time.values

for run_idx, run_dir_str in enumerate(RUN_DIRS):
    print(f"\n{'=' * 100}")
    print(f"  RUN {run_idx + 1}/{len(RUN_DIRS)}: {run_dir_str}")
    print(f"{'=' * 100}")

    RUN_DIR   = Path(run_dir_str)
    CKPT_PATH = RUN_DIR / "best_model.pt"

    if not CKPT_PATH.exists():
        print(f"  [SKIP] Checkpoint not found: {CKPT_PATH}")
        continue

    # ── Load checkpoint ─────────────────────────────────────────────────
    checkpoint = torch.load(CKPT_PATH, map_location='cpu', weights_only=False)

    model = RRDBGenerator(in_ch=4, out_ch=4, base_ch=64, num_rrdb=4)
    model.load_state_dict(checkpoint["generator"])
    model.eval()
    norm_stats = {
        'X_mean': checkpoint['X_mean'].detach().cpu().float(),
        'X_std' : checkpoint['X_std'].detach().cpu().float(),
        'Y_mean': checkpoint['Y_mean'].detach().cpu().float(),
        'Y_std' : checkpoint['Y_std'].detach().cpu().float(),
    }

    X_mu  = norm_stats['X_mean'].numpy().astype(np.float32)
    X_sig = norm_stats['X_std'].numpy().astype(np.float32)
    Y_mu  = norm_stats['Y_mean']
    Y_sig = norm_stats['Y_std']

    # ── Normalize test data with this run's stats ───────────────────────
    X_test_norm = (X_test_raw - X_mu) / (X_sig + 1e-6)
    Y_test_norm = (Y_test_raw - Y_mu.numpy().astype(np.float32)) / \
                  (Y_sig.numpy().astype(np.float32) + 1e-6)

    X_test_t = torch.from_numpy(X_test_norm).float()
    Y_test_t = torch.from_numpy(Y_test_norm).float()

    test_loader = DataLoader(
        TensorDataset(X_test_t, Y_test_t),
        batch_size=BATCH_SIZE, shuffle=False,
    )
    print(f"  Checkpoint loaded. X_test: {X_test_t.shape}, Y_test: {Y_test_t.shape}")

    # ── Run inference ───────────────────────────────────────────────────
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model  = model.to(device)

    def _match_target_hw(pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        # Align target spatial orientation to model output when H/W are swapped.
        if pred.shape[-2:] == target.shape[-2:][::-1]:
            target = target.transpose(-2, -1)
        elif pred.shape[-2:] != target.shape[-2:]:
            raise ValueError(
                f"Prediction/target spatial mismatch: pred={pred.shape}, target={target.shape}"
            )
        return target

    all_preds   = []
    all_targets = []
    with torch.no_grad():
        for X_batch, Y_batch in test_loader:
            pred = model(X_batch.to(device))
            Y_batch = _match_target_hw(pred, Y_batch.to(device))
            all_preds.append(pred.cpu())
            all_targets.append(Y_batch.cpu())

    all_preds   = torch.cat(all_preds,   dim=0)
    all_targets = torch.cat(all_targets, dim=0)

    # ── Denormalize for metrics ─────────────────────────────────────────
    y_pred_phys = denormalize(all_preds.clone(),   Y_mu, Y_sig).numpy()
    y_true_phys = denormalize(all_targets.clone(), Y_mu, Y_sig).numpy()

    # ── Print per-variable metrics ──────────────────────────────────────
    hdr = (f"\n{'Variable':<18} | {'RMSE':>8} | {'MAE':>8} | "
           f"{'Bias':>8} | {'Corr':>8} | {'Base RMSE':>14} | {'Skill':>8}")
    print(hdr)
    print('-' * 100)

    metrics_rows  = []
    formula_rows  = []

    for v_idx, v_name in enumerate(VARS):
        pred_v = y_pred_phys[:, v_idx, :, :]
        true_v = y_true_phys[:, v_idx, :, :]
        n = min(pred_v.shape[0], true_v.shape[0])
        pred_v = pred_v[:n]; true_v = true_v[:n]

        rmse_grid = np.sqrt(np.mean((pred_v - true_v) ** 2, axis=0))
        mae_grid  = np.mean(np.abs(pred_v - true_v), axis=0)
        bias_grid = np.mean(pred_v - true_v, axis=0)

        pred_diff = pred_v - np.mean(pred_v, axis=0)
        true_diff = true_v - np.mean(true_v, axis=0)
        cov = np.sum(pred_diff * true_diff, axis=0)
        with np.errstate(divide='ignore', invalid='ignore'):
            corr_grid = cov / np.sqrt(
                np.sum(pred_diff ** 2, axis=0) * np.sum(true_diff ** 2, axis=0)
            )

        rmse  = float(np.nanmean(rmse_grid))
        mae   = float(np.nanmean(mae_grid))
        bias  = float(np.nanmean(bias_grid))
        corr  = float(np.nanmean(corr_grid))
        rbase = baseline_rmse_by_var[v_name]
        skill = 1.0 - (rmse / rbase) if rbase > BASELINE_EPS else 0.0

        label = VAR_LABELS[v_idx]
        print(f"{label:<18} | {rmse:8.4f} | {mae:8.4f} | "
              f"{bias:+8.4f} | {corr:8.4f} | {rbase:14.4f} | {skill:+8.4f}")

        metrics_rows.append({
            'variable': v_name, 'label': label,
            'RMSE': rmse, 'MAE': mae, 'Bias': bias, 'Corr': corr,
            'Baseline_RMSE': rbase, 'Skill': skill,
        })
        formula_rows.append({
            'variable': v_name, 'label': label,
            'N': int(pred_v.shape[0]), 'L': int(pred_v.shape[1]),
            'W': int(pred_v.shape[2]),
            'MAE_formula13':  mae_formula_13(pred_v, true_v),
            'RMSE_formula14': rmse_formula_14(pred_v, true_v),
        })

    print('-' * 100)
    print('  Skill > 0 means model improves over bilinear interpolation baseline.')

    # ── Save metrics CSV ────────────────────────────────────────────────
    csv_path = os.path.join(str(RUN_DIR), 'metrics.csv')
    fields   = ['variable', 'label', 'RMSE', 'MAE', 'Bias', 'Corr',
                'Baseline_RMSE', 'Skill']
    with open(csv_path, 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=fields)
        w.writeheader()
        w.writerows(metrics_rows)

    formula_csv = os.path.join(str(RUN_DIR), 'metrics_formula13_14.csv')
    pd.DataFrame(formula_rows).to_csv(formula_csv, index=False)

    print(f"  Metrics saved -> {csv_path}")
    print(f"  Formula metrics saved -> {formula_csv}")

    # ── Spatial eval plot for target date ────────────────────────────────
    idx = np.where(test_times == target_date)[0]
    if len(idx) == 0:
        print(f"  [WARN] target_date {target_date} not in test set, skipping plot.")
    else:
        idx = idx[0]
        X_sample = X_test_t[[idx]].to(device)
        Y_sample = Y_test_t[[idx]]

        with torch.no_grad():
            Y_pred_sample = model(X_sample)
            Y_sample = _match_target_hw(Y_pred_sample, Y_sample.to(device)).cpu()
            Y_pred_sample = Y_pred_sample.cpu()

        X_dn    = denormalize(X_sample.cpu(), norm_stats['X_mean'],
                              norm_stats['X_std'])[0]
        Y_true_s = denormalize(Y_sample,       Y_mu, Y_sig)[0]
        Y_pred_s = denormalize(Y_pred_sample, Y_mu, Y_sig)[0]

        proj   = ccrs.PlateCarree()
        n_vars = len(VAR_LABELS_VIZ)
        col_titles = [
            "Raw FC\n(low-res 24\u00d732)",
            "SR Prediction\n(GAN 144\u00d7192)",
            "Truth\n(ERA5 144\u00d7192)",
            "Error\n(Pred \u2212 Truth)",
        ]

        fig, axes = plt.subplots(
            n_vars, 4, figsize=(22, 5 * n_vars),
            subplot_kw={"projection": proj},
        )

        for v in range(n_vars):
            raw_v       = X_dn[v].numpy()
            pred_v_plot = Y_pred_s[v].numpy()
            true_v_plot = Y_true_s[v].numpy()
            err_v       = pred_v_plot - true_v_plot

            panels = [
                (lons_lr, lats_lr, raw_v,       cmaps_field[v], FIELD_VMIN[v], FIELD_VMAX[v]),
                (lons_hr, lats_hr, pred_v_plot,  cmaps_field[v], FIELD_VMIN[v], FIELD_VMAX[v]),
                (lons_hr, lats_hr, true_v_plot,  cmaps_field[v], FIELD_VMIN[v], FIELD_VMAX[v]),
                (lons_hr, lats_hr, err_v,        "bwr",     -ERROR_LIM[v], ERROR_LIM[v]),
            ]

            for col, (lons, lats, data, cmap, lo, hi) in enumerate(panels):
                ax = axes[v, col]
                if data.shape != (len(lats), len(lons)):
                    data = data.T
                im = ax.pcolormesh(lons, lats, data, cmap=cmap, vmin=lo, vmax=hi,
                                   shading="nearest", transform=proj)
                ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
                ax.add_feature(cfeature.BORDERS,   linewidth=0.4, linestyle=":")
                ax.add_feature(cfeature.LAND,      facecolor="whitesmoke", zorder=0)
                ax.add_feature(cfeature.OCEAN,     facecolor="lightcyan",  zorder=0)
                gl = ax.gridlines(draw_labels=True, linewidth=0.3,
                                  color="gray", alpha=0.4)
                gl.top_labels = False; gl.right_labels = False
                gl.xlabel_style = {"size": 6}; gl.ylabel_style = {"size": 6}
                cb = plt.colorbar(im, ax=ax, orientation="horizontal",
                                  pad=0.05, fraction=0.046)
                cb.ax.tick_params(labelsize=7)
                if v == 0:
                    ax.set_title(col_titles[col], fontsize=10,
                                 fontweight="bold", pad=8)
                if col == 0:
                    ax.text(-0.15, 0.5, VAR_LABELS_VIZ[v],
                            transform=ax.transAxes, fontsize=10,
                            fontweight="bold", va="center", rotation=90)

            rmse_s = np.sqrt(np.mean(err_v ** 2))
            bias_s = np.mean(err_v)
            axes[v, 3].set_title(
                f"Error  |  RMSE={rmse_s:.3f}  Bias={bias_s:+.3f}",
                fontsize=9, pad=6,
            )

        fig.suptitle(
            f"GAN Pre-Up \u2014 {str(target_date)[:10]}  (6\u00d7, run {run_idx+1})",
            fontsize=14, fontweight="bold", y=1.01,
        )
        plt.tight_layout()
        fig_path = os.path.join(str(RUN_DIR),
                                f"gan_preupsample_eval_{str(target_date)[:10]}.png")
        fig.savefig(fig_path, dpi=150, bbox_inches='tight')
        print(f"  Chart saved -> {fig_path}")
        plt.show()

    # free GPU memory between runs
    del model, checkpoint
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(f"\n{'=' * 100}")
print(f"  All {len(RUN_DIRS)} runs evaluated!")
print(f"{'=' * 100}")

In [ ]:
# Notebook refactor complete.
# Metrics are saved by the evaluation cell above.
